# 01 — REF concepts

This notebook introduces the core vocabulary of the **Rapid Evaluation Framework (REF)**.
By the end you will know what a *diagnostic*, *provider*, *execution*, *metric* and *dataset* are, and how they fit together.

**Prerequisites:** none. This is the place to start.

**What you need:** an internet connection as we read from the public REF API.

## What is the REF?

The REF performs evaluation of climate datasets, much like a CI/CD pipeline runs tests against code. 
As new climate model output is published, the REF evaluates it against reference data 
and produces figures and metrics — in near-real time.

The public deployment currently evaluates CMIP6 dataset, but will include CMIP7 Assessment Fast Track data
as they become available.
Results are served from a website (<https://dashboard.climate-ref.org>) and a public API (<https://api.climate-ref.org>).

Throughout these notebooks we talk to the API via an "SDK".
This SDK allows us to make requests to the API without having to directly make HTTP requests.
The `ref_tutorials` helper package (shipped with this repository) builds the API client for us.

In [2]:
from ref_tutorials import get_client

client = get_client()
client

Client(raise_on_unexpected_status=False, _base_url='https://api.climate-ref.org', _cookies={}, _headers={}, _timeout=None, _verify_ssl=True, _follow_redirects=False, _httpx_args={}, _client=None, _async_client=None)

## Diagnostics

A **diagnostic** is a single, well-defined evaluation to understand a component of the earth system.
For example "the global mean surface temperature timeseries" or "the Atlantic overturning circulation strength".

Let's list the diagnostics the REF currently provides:

In [3]:
from climate_ref_client.api.diagnostics import diagnostics_list

diagnostics = diagnostics_list.sync(client=client).data
print(f"{len(diagnostics)} diagnostics available\n")
for diagnostic in sorted(diagnostics, key=lambda d: d.name)[:10]:
    print(f"  {diagnostic.slug:35s} {diagnostic.name}")

47 diagnostics available

  annual-cycle                        Annual Cycle Analysis
  sea-ice-area-basic                  Arctic and Antarctic Sea Ice Area Seasonal Cycle
  amoc-rapid                          Atlantic Meridional Overturning Circulation (RAPID)
  burntfractionall-gfed               Burnt Fraction (GFED)
  climate-drivers-for-fire            Climate Drivers for Fire
  climate-at-global-warming-levels    Climate at Global Warming Levels
  cloud-radiative-effects             Cloud Radiative Effects
  cloud-scatterplots-reference        Cloud Scatterplots for Reference dataset
  cloud-scatterplots-clwvi-pr         Cloud-Precipitation Scatterplots (clwvi vs pr)
  cloud-scatterplots-clivi-lwcre      Cloud-Radiation Scatterplots (clivi vs lwcre)


Every diagnostic has a human-readable `name`, a short machine-friendly `slug`, and a
`description`. Let's look at one in full:

In [4]:
example = diagnostics[0]
print("name:       ", example.name)
print("slug:       ", example.slug)
print("description:", example.description.strip())

name:        Climate at Global Warming Levels
slug:        climate-at-global-warming-levels
description: Calculate climate variables at global warming levels (e.g. 1.5C, 2C, 3C, 4C above pre-industrial).


## Providers

Diagnostics are grouped into **providers**. A **provider** is a package that knows how to compute a family of diagnostics.

For the Assessment Fast Track we use:

- ESMValTool
- PCMDI Metrics Package (PMP)
- ILAMB/IOMB

The REF orchestrates providers and does not compute diagnostics itself.
Each provider is a thin wrapper around the upstream diagnostics package.

Each diagnostic tells you which provider it belongs to:

In [5]:
providers = {d.provider.slug: d.provider.name for d in diagnostics}
for slug, name in sorted(providers.items()):
    count = sum(1 for d in diagnostics if d.provider.slug == slug)
    print(f"  {slug:20s} {name}  ({count} diagnostics)")

  esmvaltool           ESMValTool  (24 diagnostics)
  ilamb                ILAMB  (13 diagnostics)
  pmp                  PMP  (10 diagnostics)


## Datasets

A diagnostic needs **datasets** to run against: the climate model output under evaluation, and the reference data it is compared to.
The REF tracks which datasets are available and which diagnostics have already been run.

For the public API we do not handle raw CMIP6 datasets directly the evaluations have already
been run. We will see locally fetched datasets in notebook 04.

## Executions

An **execution** is one run of a diagnostic against one specific group of datasets.

A single diagnostic is typically executed many times. This generally depends what is being calculated, but this is often once per model variant.
Each of these individual groups is an *execution group*.
If any datasets in an execution group change, then a new execution is performed.

Here are the execution groups of one diagnostic:

In [6]:
with_executions = next(d for d in diagnostics if d.execution_groups)
print(f"Diagnostic: {with_executions.name}")
print(f"Execution groups: {len(with_executions.execution_groups)}")

Diagnostic: Climate at Global Warming Levels
Execution groups: 4


## Metrics

An execution produces output. That output comes in two main shapes:

- **Metric values** — numbers that summarise a property of a model. 
  A *scalar* is a single number (e.g. a root-mean-square error) or a *series* is a number per index point
  (e.g. a seasonal cycle).
- **Files** — NetCDF data and figures for deeper analysis or custom plotting.

The REF ingests all of these outputs into its database so they can be queried by the API.

Notebook 02 shows how to retrieve metric values, and notebook 03 turns them into a figure.

## Recap

| Term                | Meaning                                                                       |
| ------------------- | ----------------------------------------------------------------------------- |
| **Provider**        | A package that computes a family of diagnostics (ESMValTool, PMP, ILAMB, ...) |
| **Diagnostic**      | A single well-defined evaluation                                              |
| **Dataset**         | Climate model output or reference data a diagnostic runs against              |
| **Execution**       | One run of a diagnostic against one group of datasets                         |
| **Metric value**    | A scalar or series result summarising a model                                 |

**Next:** [02 — Querying the REF API](02-querying-the-api.ipynb).